In [149]:
import xml.etree.ElementTree as ET
from collections import defaultdict

# Parse the alignment file
alignment_tree = ET.parse('alignment.xml')  
alignment_root = alignment_tree.getroot()

# Extract alignment pairs (en_id -> ml_id)
alignments = []
for link in alignment_root.findall('.//link'):
    xtargets = link.get('xtargets')
    en_ids, ml_ids = xtargets.split(';')
    en_ids = [int(id) for id in en_ids.split()]
    ml_ids = [int(id) for id in ml_ids.split()]
    alignments.append((en_ids, ml_ids))

# Parse English and Malayalam files
def parse_text_file(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()
    sentences = {}
    for s in root.findall('s'):
        sent_id = int(s.get('id'))
        words = [w.text for w in s.findall('w') if w.text is not None]
        sentences[sent_id] = ' '.join(words)
    return sentences

english_sents = parse_text_file(r'en\mozilla-i10n-v1.xml')
malayalam_sents = parse_text_file(r'ml\mozilla-i10n-v1.xml')

# Create aligned dataset
dataset = []
for en_ids, ml_ids in alignments:
    # Get all aligned English sentences (join if multiple)
    en_text = ' '.join(english_sents.get(en_id, '') for en_id in en_ids)
    # Get all aligned Malayalam sentences
    ml_text = ' '.join(malayalam_sents.get(ml_id, '') for ml_id in ml_ids)
    
    if en_text and ml_text:  # Only add if both exist
        dataset.append((en_text.strip(), ml_text.strip()))

# Save as CSV 
import csv
with open('aligned_dataset.csv', 'w', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['English', 'Malayalam'])  # Header
    writer.writerows(dataset)

In [150]:
import pandas as pd
parallel_corpus = pd.read_csv('aligned_dataset.csv')
parallel_corpus.head(20)

,English,Malayalam
0,Tagalog,ത ഗ ല ഗ്‌
1,Tags,ട ഗുകള്‍
2,Tags,റ്റ ഗുകള്‍
3,Tags,ട ഗുകള്‍
4,Tags :,ട ഗുകള്‍ :
5,Tags Added,ട ഗുകൾ ച ർത്തു
6,Tags are limited to 25 characters,ട ഗുകള്‍ 25 അക്ഷരങ്ങള ല ക്ക് പര മ തപ്പ ടുത്ത യ...
7,Tahitian,തഹ ത യന്‍
8,Taiwan,ത യ്‌വ ന്‍
9,Tajik,ത ജ ക്ക്‌


In [152]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Load the parallel corpus and drop rows with missing values
df = pd.read_csv("aligned_dataset.csv").dropna(subset=['English', 'Malayalam'])

# Verify no NaN values remain
print(f"Rows after cleaning: {len(df)}")

# Vectorize English text (replace empty strings if any)
df['English'] = df['English'].fillna('').astype(str)  # Ensure all are strings
tfidf_en = TfidfVectorizer(max_features=5000)  
X_en = tfidf_en.fit_transform(df['English'])

# Vectorize Malayalam text
df['Malayalam'] = df['Malayalam'].fillna('').astype(str)
tfidf_ml = TfidfVectorizer(max_features=5000)
X_ml = tfidf_ml.fit_transform(df['Malayalam'])

# Save the vectorized data
import numpy as np
np.save("X_en_tfidf.npy", X_en.toarray())  
np.save("X_ml_tfidf.npy", X_ml.toarray())

Rows after cleaning: 17269


In [157]:
X = np.load("X_en_tfidf.npy")
y = np.load("X_ml_tfidf.npy")

In [159]:
import pickle
from sklearn.linear_model import Ridge

model = Ridge(alpha=1.0)
model.fit(X , y)


Ridge()

In [171]:
import pickle

# Save English TF-IDF
with open('tfidf_en.pkl', 'wb') as f:
    pickle.dump(tfidf_en, f)

# Save Malayalam TF-IDF
with open('tfidf_ml.pkl', 'wb') as f:
    pickle.dump(tfidf_ml, f)

# Save the Model
with open('translator.pkl' , 'wb') as f:
    pickle.dump(model , f)

with open('malayalam_corpus.pkl' , 'wb') as f:
    pickle.dump(df['Malayalam'],f)

In [175]:
import pickle
from keras.models import load_model
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def load_and_translate(sentence):
    # Load the trained Keras model
    with open("translator.pkl" , 'rb') as f:
        model = pickle.load(f)

    # Load the TF-IDF vectorizers
    with open("tfidf_en.pkl", "rb") as f:
        tfidf_en = pickle.load(f)

    with open("tfidf_ml.pkl", "rb") as f:
        tfidf_ml = pickle.load(f)

    with open("malayalam_corpus.pkl", "rb") as f:
        ml_corpus = pickle.load(f)

    # Vectorize the English sentence
    en_vector = tfidf_en.transform([sentence])

    # Predict the Malayalam vector
    predicted_vector = model.predict(en_vector)

    # Find closest match in Malayalam corpus using cosine similarity
    similarities = cosine_similarity(predicted_vector, tfidf_ml.transform(ml_corpus))
    most_similar_index = np.argmax(similarities)

    return ml_corpus[most_similar_index]


In [183]:
translation = load_and_translate("The smell of cold beer is damp.")
print("Translated:", translation)

Translated: ച ല പരസ്യങ്ങൾക്കുള്ള ൽ ന ങ്ങള ഓൺല ന യ പ ന്തുടരുന്ന അദൃശ്യമ യ ട്ര ക്കറുകളുണ്ട് . എന്ത് മ ശ ക ര്യമ ണത് . ഞങ്ങൾക്കറ യ . അത ന ല ണ് ഞങ്ങളുട ശക്തമ യ ഉപകരണ അവയ ന ർദ്ദയ തടയുന്നത് .
